In [1]:
from pathlib import Path

DATA_DIR = Path(r"C:\Users\hp\Downloads\archive (3)\data\data")

print(DATA_DIR.exists())
print(DATA_DIR)

True
C:\Users\hp\Downloads\archive (3)\data\data


In [ ]:
categories = [folder.name for folder in DATA_DIR.iterdir() if folder.is_dir()] #عرض أسماء التصنيفات

print("Number of categories:", len(categories))
print(categories)

Number of categories: 24
['ACCOUNTANT', 'ADVOCATE', 'AGRICULTURE', 'APPAREL', 'ARTS', 'AUTOMOBILE', 'AVIATION', 'BANKING', 'BPO', 'BUSINESS-DEVELOPMENT', 'CHEF', 'CONSTRUCTION', 'CONSULTANT', 'DESIGNER', 'DIGITAL-MEDIA', 'ENGINEERING', 'FINANCE', 'FITNESS', 'HEALTHCARE', 'HR', 'INFORMATION-TECHNOLOGY', 'PUBLIC-RELATIONS', 'SALES', 'TEACHER']


In [ ]:
#تجهيز جدول فيه كل ملفات الـ PDF
import pandas as pd

rows = []

for category_folder in DATA_DIR.iterdir():
    if category_folder.is_dir():
        category = category_folder.name
        
        for pdf_file in category_folder.glob("*.pdf"):
            rows.append({
                "file_path": str(pdf_file),
                "category": category
            })

df = pd.DataFrame(rows)

df.head()

,file_path,category
0,C:\Users\hp\Downloads\archive (3)\data\data\AC...,ACCOUNTANT
1,C:\Users\hp\Downloads\archive (3)\data\data\AC...,ACCOUNTANT
2,C:\Users\hp\Downloads\archive (3)\data\data\AC...,ACCOUNTANT
3,C:\Users\hp\Downloads\archive (3)\data\data\AC...,ACCOUNTANT
4,C:\Users\hp\Downloads\archive (3)\data\data\AC...,ACCOUNTANT


In [ ]:
df.shape #كم CV

(2484, 2)

In [5]:
df["category"].value_counts()

category
INFORMATION-TECHNOLOGY    120
BUSINESS-DEVELOPMENT      120
ACCOUNTANT                118
ADVOCATE                  118
CHEF                      118
ENGINEERING               118
FINANCE                   118
AVIATION                  117
FITNESS                   117
SALES                     116
HEALTHCARE                115
CONSULTANT                115
BANKING                   115
CONSTRUCTION              112
PUBLIC-RELATIONS          111
HR                        110
DESIGNER                  107
ARTS                      103
TEACHER                   102
APPAREL                    97
DIGITAL-MEDIA              96
AGRICULTURE                63
AUTOMOBILE                 36
BPO                        22
Name: count, dtype: int64

In [ ]:
#استخراج النص من الـ PDF
from pypdf import PdfReader
from tqdm import tqdm

def extract_text_from_pdf(pdf_path):
    try:
        reader = PdfReader(pdf_path)
        text = ""

        for page in reader.pages:
            page_text = page.extract_text()
            if page_text:
                text += page_text + " "

        return " ".join(text.split())

    except Exception:
        return ""

tqdm.pandas()

df["raw_text"] = df["file_path"].progress_apply(extract_text_from_pdf)

df.head()

100%|██████████| 2484/2484 [22:42<00:00,  1.82it/s]


,file_path,category,raw_text
0,C:\Users\hp\Downloads\archive (3)\data\data\AC...,ACCOUNTANT,ACCOUNTANT Summary Financial Accountant specia...
1,C:\Users\hp\Downloads\archive (3)\data\data\AC...,ACCOUNTANT,STAFF ACCOUNTANT Summary Highly analytical and...
2,C:\Users\hp\Downloads\archive (3)\data\data\AC...,ACCOUNTANT,ACCOUNTANT Professional Summary To obtain a po...
3,C:\Users\hp\Downloads\archive (3)\data\data\AC...,ACCOUNTANT,SENIOR ACCOUNTANT Experience Company Name June...
4,C:\Users\hp\Downloads\archive (3)\data\data\AC...,ACCOUNTANT,SENIOR ACCOUNTANT Professional Summary Senior ...


In [ ]:
#فحص النصوص الفاضية أو الضعيفة
df["text_length"] = df["raw_text"].apply(len)

df[["category", "text_length", "raw_text"]].head()

,category,text_length,raw_text
0,ACCOUNTANT,24153,ACCOUNTANT Summary Financial Accountant specia...
1,ACCOUNTANT,7488,STAFF ACCOUNTANT Summary Highly analytical and...
2,ACCOUNTANT,4742,ACCOUNTANT Professional Summary To obtain a po...
3,ACCOUNTANT,5917,SENIOR ACCOUNTANT Experience Company Name June...
4,ACCOUNTANT,5561,SENIOR ACCOUNTANT Professional Summary Senior ...


In [8]:
df["text_length"].describe()

count     2484.000000
mean      5933.672705
std       2626.836960
min          0.000000
25%       4829.000000
50%       5559.500000
75%       6864.000000
max      35115.000000
Name: text_length, dtype: float64

In [ ]:
#تنظيف الداتا
empty_count = (df["text_length"] == 0).sum()
short_count = (df["text_length"] < 100).sum()

print("Empty PDFs:", empty_count)
print("Short PDFs less than 100 chars:", short_count)

Empty PDFs: 1
Short PDFs less than 100 chars: 1


In [ ]:
#حذف الملف الفاضي/الضعيف
df_clean = df[df["text_length"] > 100].copy()

print("Before cleaning:", df.shape)
print("After removing empty/short resumes:", df_clean.shape)

Before cleaning: (2484, 4)
After removing empty/short resumes: (2483, 4)


In [ ]:
#تنظيف النص
import re

def clean_resume_text(text):
    text = str(text).lower()

    # remove emails
    text = re.sub(r'\S+@\S+', ' ', text)

    # remove urls
    text = re.sub(r'http\S+|www\S+', ' ', text)

    # remove phone numbers
    text = re.sub(r'\+?\d[\d\s\-\(\)]{7,}\d', ' ', text)

    # keep only English letters and spaces
    text = re.sub(r'[^a-zA-Z\s]', ' ', text)

    # remove extra spaces
    text = re.sub(r'\s+', ' ', text).strip()

    return text

df_clean["clean_text"] = df_clean["raw_text"].apply(clean_resume_text)

df_clean[["category", "raw_text", "clean_text"]].head()

,category,raw_text,clean_text
0,ACCOUNTANT,ACCOUNTANT Summary Financial Accountant specia...,accountant summary financial accountant specia...
1,ACCOUNTANT,STAFF ACCOUNTANT Summary Highly analytical and...,staff accountant summary highly analytical and...
2,ACCOUNTANT,ACCOUNTANT Professional Summary To obtain a po...,accountant professional summary to obtain a po...
3,ACCOUNTANT,SENIOR ACCOUNTANT Experience Company Name June...,senior accountant experience company name june...
4,ACCOUNTANT,SENIOR ACCOUNTANT Professional Summary Senior ...,senior accountant professional summary senior ...


In [ ]:
#فحص طول النص بعد التنظيف
df_clean["clean_text_length"] = df_clean["clean_text"].apply(len)

df_clean["clean_text_length"].describe()

count     2483.000000
mean      5642.345550
std       2507.566373
min        628.000000
25%       4596.000000
50%       5263.000000
75%       6518.000000
max      33877.000000
Name: clean_text_length, dtype: float64

In [ ]:
#حذف التكرار
before_duplicates = df_clean.shape[0]

df_clean = df_clean.drop_duplicates(subset=["clean_text"]).copy()

after_duplicates = df_clean.shape[0]

print("Before duplicates removal:", before_duplicates)
print("After duplicates removal:", after_duplicates)
print("Removed duplicates:", before_duplicates - after_duplicates)

Before duplicates removal: 2483
After duplicates removal: 2481
Removed duplicates: 2


In [ ]:
#حفظ الملف 
df_clean.to_csv("resume_clean_dataset.csv", index=False, encoding="utf-8-sig")

In [ ]:
#تاكد انو انحفظ الملف
import os

print(os.path.exists("resume_clean_dataset.csv"))

True


In [ ]:
#فحص الأعمدة
df_clean.columns

Index(['file_path', 'category', 'raw_text', 'text_length', 'clean_text',
       'clean_text_length'],
      dtype='object')